# 6-1 Notebook 코드를 Python 모듈로 분리하기

## 학습 목표

- Notebook에 작성한 코드를 기능별 Python 파일로 나누는 기준을 이해한다.
- 실무형 프로젝트 구조로 바꾸는 방법을 익힌다.
- 이 노트북은 리팩터링 설명과 예시 중심으로 구성한다.

## 1. 왜 모듈화가 필요한가?

Notebook은 학습과 실험에 좋지만, 실제 앱이나 서비스로 만들 때는 코드 관리가 어렵다.

모듈화의 핵심은 파일을 많이 만드는 것이 아니라, **역할을 분리하는 것**이다.

## 2. 분리 기준

| Notebook 코드 | 분리할 파일 | 역할 |
|---|---|---|
| 경로, 모델명, 설정값 | `config.py` | 전역 설정 |
| PDF 로딩 함수 | `document_loader.py` | 문서 로딩 |
| 청크 분할 함수 | `chunker.py` | 문서 분할 |
| Qdrant 생성 코드 | `vectorstore.py` | 벡터DB 관리 |
| 질문 분류, 재작성 | `retriever.py` | 검색 보조 |
| 프롬프트 | `prompts.py` | 프롬프트 관리 |
| State 정의 | `graph_state.py` | LangGraph 상태 |
| Node 함수 | `graph_nodes.py` | 그래프 노드 |
| Graph 연결 | `graph_builder.py` | 그래프 생성 |
| 실행 코드 | `main.py` | 앱 진입점 |

## 3. `config.py` 예시

In [8]:
# src/config.py

from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv(override=True, dotenv_path="../../.env")


PDF_PATH = Path('data') / "공직자_민원응대_핵심_매뉴얼.pdf"

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")

CHUNK_SIZE = 700
CHUNK_OVERLAP = 120
TOP_K = 4

## 4. `document_loader.py` 예시

In [9]:
# src/document_loader.py

import re
from pathlib import Path
from pypdf import PdfReader
from langchain_core.documents import Document

def clean_text(text: str) -> str:
    text = text.replace("\\x00", " ")
    text = re.sub(r"[ \\t]+", " ", text)
    text = re.sub(r"\\n{3,}", "\\n\\n", text)
    return text.strip()

def load_pdf_pages(pdf_path: Path) -> list[Document]:
    reader = PdfReader(str(pdf_path))
    docs = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = clean_text(page.extract_text() or "")
        if text:
            docs.append(Document(
                page_content=text,
                metadata={"source": pdf_path.name, "page": page_number}
            ))

    return docs

## 5. `graph_builder.py` 예시

In [10]:
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    """
    현재 위치에서 위/아래를 탐색하여 graph_rag 폴더가 있는 프로젝트 루트를 찾는다.
    """
    start = start.resolve()

    # 1) 현재 위치와 상위 폴더들에서 graph_rag 찾기
    for path in [start, *start.parents]:
        if (path / "graph_rag").is_dir():
            return path

    # 2) 현재 위치 하위에서 graph_rag 찾기
    candidates = list(start.rglob("graph_rag"))
    candidates = [p for p in candidates if p.is_dir()]

    if candidates:
        return candidates[0].parent

    raise FileNotFoundError("graph_rag 폴더를 찾지 못했다. 현재 작업 경로를 확인해야 한다.")

BASE_DIR = find_project_root(Path.cwd())

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

print("프로젝트 루트:", BASE_DIR)
print("graph_rag 존재 여부:", (BASE_DIR / "graph_rag").exists())
print("sys.path[0]:", sys.path[0])

프로젝트 루트: C:\Users\magpi\langgraph_modular_rag_v1\6.modular_rag_manual
graph_rag 존재 여부: True
sys.path[0]: C:\Users\magpi\langgraph_modular_rag_v1\6.modular_rag_manual


In [11]:
# graph_rag/graph_builder.py

from langgraph.graph import StateGraph, START, END
from graph_rag.graph_state import ModularRAGState
from graph_rag.graph_nodes import (
    classify_question_node,
    retrieve_manual_node,
    organize_evidence_node,
    generate_answer_node,
    validate_answer_node,
)

def build_graph():
    graph = StateGraph(ModularRAGState)

    graph.add_node("classify_question", classify_question_node)
    graph.add_node("retrieve_manual", retrieve_manual_node)
    graph.add_node("organize_evidence", organize_evidence_node)
    graph.add_node("generate_answer", generate_answer_node)
    graph.add_node("validate_answer", validate_answer_node)

    graph.add_edge(START, "classify_question")
    graph.add_edge("classify_question", "retrieve_manual")
    graph.add_edge("retrieve_manual", "organize_evidence")
    graph.add_edge("organize_evidence", "generate_answer")
    graph.add_edge("generate_answer", "validate_answer")
    graph.add_edge("validate_answer", END)

    return graph.compile()

## 6. 그래프 그리기

In [ ]:
from graph_rag.graph_builder import build_graph
from IPython.display import Image, display

app = build_graph()
display(Image(app.get_graph().draw_mermaid_png()))

## 7. 최종 실행 흐름


In [13]:

from graph_rag.graph_builder import build_graph

app = build_graph()

result = app.invoke({
    "question": "민원인이 반복 전화를 하면 어떻게 해야 하나요?"
})

print(result["answer"])

1. 핵심 대응
   - 반복 전화를 하는 민원인에게는 통화 지속이 곤란하다는 점을 안내하고, 필요 시 통화를 종료합니다. 또한, 반복 민원에 대해서는 종결 처리할 수 있음을 설명합니다.

2. 단계별 조치
   - 민원인이 동일한 내용을 3회 이상 반복할 경우:
     1. 통화 중 충분한 설명을 제공하고, 통화 지속이 곤란하다는 점을 안내합니다.
     2. 반복 민원에 대해 2회 이상 결과를 통지한 후, 이후 민원은 종결 처리할 수 있음을 설명합니다.
     3. 종결 처리 시 민원인에게 통지하고, 이후 반복 민원은 처리 결과 회신 없이 종결합니다.

3. 사용할 수 있는 안내 표현
   - "죄송하지만, 동일한 내용의 민원은 이미 처리된 사항입니다."
   - "현재 통화가 길어지고 있어, 용건 위주로 말씀해 주시면 감사하겠습니다."
   - "이와 같은 반복 민원은 종결 처리될 수 있습니다."

4. 담당자 보호조치
   - 민원인이 폭언이나 협박을 할 경우 즉시 통화를 종료하고, 해당 내용을 기록하여 증거를 확보합니다. 필요 시 후속 조치를 취합니다.

5. 주의사항
   - 민원인의 감정이 격해질 수 있으므로, 차분하고 정중한 태도로 대응해야 합니다.
   - 반복 민원에 대한 종결 처리 시, 민원인에게 충분한 설명을 제공하여 오해가 없도록 해야 합니다.


## 핵심 정리

- Notebook은 학습용이다.
- `src/` 구조는 앱 개발과 유지보수용이다.
- Modular RAG는 기능을 분리하고, LangGraph는 그 기능들을 흐름으로 연결한다.